# DiffBIR Stage 1 (Restoration U-Net) — Kaggle Notebook

Kaggle port of the Colab notebook -- same tested code, adapted for Kaggle's environment (no Drive mount; uses `/kaggle/working` for outputs and Kaggle's Datasets mechanism for cross-session persistence instead).

**Before running anything:**
1. On the right sidebar: **Settings > Accelerator > GPU T4 x2** (or P100)
2. On the right sidebar: **Settings > Internet > On** (required for pip installs, git clone, and the VOC2012 download -- off by default on Kaggle)

**How persistence works here (different from Colab's Drive mount):**
- `/kaggle/working/` is your writable scratch space -- checkpoints, masks, and data all go here during a session
- When you click **Save Version** (top right), Kaggle commits the notebook and saves everything in `/kaggle/working/` as that version's **Output**
- To continue training in a **new** session (e.g. after hitting the weekly GPU quota or a 12-hour session limit): open a new session on this notebook, click **Add Data > Notebook Output Files**, select your own previous version, and its files appear read-only under `/kaggle/input/<notebook-name>/`. Copy the latest checkpoint from there back into `/kaggle/working/` and pass it to `--resume`.


## 1. Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected — go to Settings (right sidebar) > Accelerator > GPU T4 x2.')


## 2. Install dependencies

In [ ]:
!pip install -q pillow opencv-python-headless scikit-image scipy pandas
import torch, torchvision
print('torch', torch.__version__)
print('torchvision', torchvision.__version__)


## 3. Recreate the project files

These write out the exact same tested files used on Colab/locally -- includes the screen-blend fix (white/light damage marks) and the `--amp` / `--steps-per-epoch` speed options.

Working directory is `/kaggle/working/` so everything here persists into the notebook's Output when you Save Version.

In [ ]:
import os
os.chdir('/kaggle/working')
os.makedirs('models', exist_ok=True)
os.makedirs('data', exist_ok=True)


In [ ]:
%%writefile models/__init__.py



In [ ]:
%%writefile models/blocks.py
"""
Building blocks for the Stage 1 restoration network.

Note the deliberate architectural difference from the VAE scaffold used for
the GAN+latent-translation method: Stage 1 here is a *deterministic*
restoration network with U-Net-style skip connections, not a VAE with a
compressive stochastic bottleneck. That's an intentional choice, not an
oversight -- Stage 1's job is "remove degradation while staying as
faithful as possible to the input," and skip connections are what let fine
detail (edges, texture) bypass the bottleneck entirely rather than being
forced through a compressed latent representation. The original DiffBIR
paper uses SwinIR (a transformer-based restorer) for this stage; a
convolutional U-Net plays the same functional role at a fraction of the
compute, which is a reasonable and defensible scope reduction for a thesis
rather than a full from-scratch SwinIR reproduction.
"""

import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    """Two convs + norm + activation, resolution-preserving. The basic unit
    used at every U-Net stage (encoder, bottleneck, and decoder)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(out_channels, affine=True),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DownBlock(nn.Module):
    """ConvBlock followed by a strided-conv downsample. Returns both the
    pre-downsample features (kept as a skip connection) and the
    downsampled output (passed deeper into the encoder)."""

    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.downsample = nn.Conv2d(out_channels, out_channels, kernel_size=4, stride=2, padding=1)

    def forward(self, x: torch.Tensor):
        skip = self.conv(x)
        down = self.downsample(skip)
        return down, skip


class UpBlock(nn.Module):
    """Upsamples, concatenates the matching encoder skip connection, then
    fuses with a ConvBlock. This is the actual mechanism that lets Stage 1
    stay faithful to fine input detail rather than smoothing everything
    through the bottleneck."""

    def __init__(self, in_channels: int, skip_channels: int, out_channels: int):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, in_channels, kernel_size=4, stride=2, padding=1)
        self.conv = ConvBlock(in_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.upsample(x)
        # Guard against off-by-one size mismatches from odd input dimensions
        if x.shape[-2:] != skip.shape[-2:]:
            x = torch.nn.functional.interpolate(x, size=skip.shape[-2:], mode="nearest")
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


In [ ]:
%%writefile models/restoration_net.py
"""
Stage 1 restoration network: a convolutional U-Net that maps a damaged
(scratches/smut-composited) photo to a restored one.

This is deliberately simpler than DiffBIR's own Stage 1 (which uses
SwinIR, a much heavier transformer-based restorer trained on a broad,
generic blur/noise/JPEG/downsampling degradation model). Two scope
decisions worth being explicit about in your thesis writeup:

1. Architecture: U-Net instead of SwinIR. Same functional role (faithful,
   detail-preserving degradation removal), far less compute. This is a
   standard, well-understood restoration architecture in its own right
   (used across denoising/inpainting literature), not a shortcut invented
   for this project.

2. Degradation model: trained specifically on YOUR FilmDamageSimulator
   scratches/smut compositing, not DiffBIR's generic blur/noise/JPEG/
   downsampling pipeline. This is actually a *better* fit for your thesis
   question (how well does this restoration paradigm handle these specific
   damage types) than reproducing their generic degradation model would be.
"""

import torch
import torch.nn as nn

from models.blocks import ConvBlock, DownBlock, UpBlock


class RestorationUNet(nn.Module):
    def __init__(self, in_channels: int = 3, out_channels: int = 3,
                 base_channels: int = 64, n_downsample: int = 4, max_channels: int = 512):
        super().__init__()

        # Build the channel sequence for each encoder stage, e.g. for
        # base_channels=64, n_downsample=4: [64, 128, 256, 512, 512]
        channels = [min(base_channels * (2 ** i), max_channels) for i in range(n_downsample + 1)]

        self.down_blocks = nn.ModuleList()
        in_ch = in_channels
        for out_ch in channels[:-1]:
            self.down_blocks.append(DownBlock(in_ch, out_ch))
            in_ch = out_ch

        self.bottleneck = nn.Sequential(
            ConvBlock(channels[-2], channels[-1]),
            ConvBlock(channels[-1], channels[-1]),
        )

        self.up_blocks = nn.ModuleList()
        up_in_ch = channels[-1]
        for skip_ch in reversed(channels[:-1]):
            self.up_blocks.append(UpBlock(up_in_ch, skip_ch, skip_ch))
            up_in_ch = skip_ch

        self.final_conv = nn.Sequential(
            nn.Conv2d(channels[0], out_channels, kernel_size=3, padding=1),
            nn.Tanh(),  # output in [-1, 1], matching the dataset's normalization
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips = []
        for down in self.down_blocks:
            x, skip = down(x)
            skips.append(skip)

        x = self.bottleneck(x)

        for up, skip in zip(self.up_blocks, reversed(skips)):
            x = up(x, skip)

        return self.final_conv(x)


def restoration_loss(pred: torch.Tensor, target: torch.Tensor, l1_weight: float = 1.0):
    """
    L1 reconstruction loss, matching DiffBIR's own design reasoning for
    this stage: a regression loss here produces a faithful-but-slightly-
    smoothed output, and that's intentional -- Stage 2 (the diffusion
    prior, built separately) is what adds sharp detail back on top of
    this. Don't be tempted to add heavy perceptual/adversarial losses here
    to make Stage 1 alone look sharper; that would blur the separation of
    concerns the two-stage design is built around.
    """
    l1 = torch.nn.functional.l1_loss(pred, target)
    return l1_weight * l1, l1


In [ ]:
%%writefile data/__init__.py



In [ ]:
%%writefile data/degraded_pair_dataset.py
"""
Dataset for Stage 1 training: pairs of (damaged, clean) images, where the
damage comes from YOUR existing FilmDamageSimulator mask pool (generated
via generate_synthetic_only.py) rather than DiffBIR's generic synthetic
blur/noise/JPEG degradation pipeline.

Masks are composited onto clean images on the fly (mask value 255 = clean,
toward 0 = damaged), so a given clean image can pair with a different
random mask each epoch -- more effective training variety than
pre-generating a fixed set of damaged/clean pairs once.

Blend mode matches composite_damage.py's two options:
  - "screen" (default): damage LIGHTENS toward white. Physically realistic
    for scratches/abrasion, where the print's emulsion is scraped away and
    the lighter paper base shows through.
  - "multiply": damage DARKENS toward black. More appropriate for damage
    that deposits dark material (soot/smut, heavy dirt, mold staining).
Since a mixed mask (e.g. scratches + smut generated together) doesn't track
which pixel came from which damage type, this is a dataset-wide setting --
if you want type-appropriate blending for a mixed mask pool, generate and
composite scratches and smut as separate mask batches with different
--blend-mode settings instead of one mixed pool.
"""

import os
import random

from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import torchvision.transforms.functional as TF


class DegradedPairDataset(Dataset):
    """
    Expects:
        clean_dir/   -- folder of clean photos (e.g. a VOC2012 subset)
        masks_dir/   -- folder of grayscale masks from generate_synthetic_only.py
                         (mask_*.png; NOT the binarised_mask_*.png variants --
                         those are thresholded and lose the soft edges that
                         make compositing look natural)
    """

    def __init__(self, clean_dir: str, masks_dir: str, image_size: int = 256, augment: bool = True,
                 blend_mode: str = "screen"):
        if blend_mode not in ("screen", "multiply"):
            raise ValueError(f"blend_mode must be 'screen' or 'multiply', got '{blend_mode}'")
        self.clean_dir = clean_dir
        self.masks_dir = masks_dir
        self.image_size = image_size
        self.augment = augment
        self.blend_mode = blend_mode

        valid_ext = (".jpg", ".jpeg", ".png")
        self.clean_files = [f for f in os.listdir(clean_dir) if f.lower().endswith(valid_ext)]
        self.mask_files = [f for f in os.listdir(masks_dir)
                            if f.lower().endswith(".png") and not f.startswith("binarised_mask")]

        if len(self.clean_files) == 0:
            raise ValueError(f"No clean images found in {clean_dir}")
        if len(self.mask_files) == 0:
            raise ValueError(f"No usable masks found in {masks_dir} "
                              f"(looking for mask_*.png, excluding binarised_mask_*.png)")

        load_size = int(image_size * 1.12)
        self.clean_resize = T.Resize(load_size)
        self.image_size_final = image_size

    def __len__(self):
        return len(self.clean_files)

    def _load_clean(self, idx):
        path = os.path.join(self.clean_dir, self.clean_files[idx])
        img = Image.open(path).convert("RGB")
        return self.clean_resize(img)

    def _load_random_mask(self):
        path = os.path.join(self.masks_dir, random.choice(self.mask_files))
        mask = Image.open(path).convert("L")  # single-channel grayscale
        return mask

    def _synchronized_crop_and_flip(self, clean_img, mask_img):
        """Applies the SAME random crop and flip to both the clean image
        and the mask, so the damage stays spatially aligned with the
        content it's composited onto."""
        # Resize mask to match the (already resized) clean image
        mask_img = mask_img.resize(clean_img.size, Image.BILINEAR)

        if self.augment:
            i, j, h, w = T.RandomCrop.get_params(clean_img, output_size=(self.image_size_final, self.image_size_final))
            clean_img = TF.crop(clean_img, i, j, h, w)
            mask_img = TF.crop(mask_img, i, j, h, w)

            if random.random() < 0.5:
                clean_img = TF.hflip(clean_img)
                mask_img = TF.hflip(mask_img)
            # Masks (unlike photo content) are safe to rotate freely --
            # scratches/smut don't have a "correct" orientation the way a
            # photo of a person or building does.
            if random.random() < 0.5:
                angle = random.choice([90, 180, 270])
                mask_img = TF.rotate(mask_img, angle)
        else:
            clean_img = TF.center_crop(clean_img, (self.image_size_final, self.image_size_final))
            mask_img = TF.center_crop(mask_img, (self.image_size_final, self.image_size_final))

        return clean_img, mask_img

    def __getitem__(self, idx):
        try:
            clean_img = self._load_clean(idx)
            mask_img = self._load_random_mask()
        except Exception:
            return self.__getitem__(random.randrange(len(self)))

        clean_img, mask_img = self._synchronized_crop_and_flip(clean_img, mask_img)

        clean_tensor = TF.to_tensor(clean_img)          # [0, 1], shape (3, H, W)
        mask_tensor = TF.to_tensor(mask_img)             # [0, 1], shape (1, H, W)

        # Composite damage onto the clean image using the configured blend mode.
        if self.blend_mode == "screen":
            # Lightens toward white at damaged (low-mask) pixels.
            damaged_tensor = 1.0 - (1.0 - clean_tensor) * mask_tensor
        else:  # "multiply"
            # Darkens toward black at damaged (low-mask) pixels.
            damaged_tensor = clean_tensor * mask_tensor

        # Normalize both to [-1, 1] to match the restoration network's Tanh output
        normalize = T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        clean_tensor = normalize(clean_tensor)
        damaged_tensor = normalize(damaged_tensor)

        return {"damaged": damaged_tensor, "clean": clean_tensor}


def denormalize(tensor: torch.Tensor) -> torch.Tensor:
    """Inverse of the Normalize(mean=0.5, std=0.5) above, for saving/viewing."""
    return (tensor * 0.5 + 0.5).clamp(0, 1)


In [ ]:
%%writefile train_stage1_restoration.py
"""
Train the Stage 1 restoration network on (damaged, clean) pairs, where
damage comes from your FilmDamageSimulator mask pool.

Usage:
    python train_stage1_restoration.py \
        --clean-dir ./voc_subset --masks-dir ./generated_masks \
        --epochs 50 --batch-size 8 --image-size 256 \
        --out-dir ./runs/stage1_restoration
"""

import argparse
import os
import time

import torch
from torch.utils.data import DataLoader, RandomSampler
import torchvision.utils as vutils

from models.restoration_net import RestorationUNet, restoration_loss
from data.degraded_pair_dataset import DegradedPairDataset, denormalize


def format_duration(seconds):
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f"{hours}h {minutes}m {secs}s"
    if minutes:
        return f"{minutes}m {secs}s"
    return f"{secs}s"


def save_comparison_grid(model, batch, out_path, device, max_images=6):
    """Saves [damaged input | model output | clean target] so you can
    visually confirm the network is actually removing damage, not just
    minimizing loss in some degenerate way (e.g. blurring everything)."""
    model.eval()
    with torch.no_grad():
        n = min(max_images, batch["damaged"].shape[0])
        damaged = batch["damaged"][:n].to(device)
        clean = batch["clean"][:n].to(device)
        restored = model(damaged)
        comparison = torch.cat([denormalize(damaged), denormalize(restored), denormalize(clean)], dim=0)
        # nrow=n (not a fixed max_images) guarantees each group of n images
        # forms exactly one row, regardless of how the actual batch size
        # compares to max_images.
        vutils.save_image(comparison, out_path, nrow=n)
    model.train()


def main():
    parser = argparse.ArgumentParser(description="Train the Stage 1 restoration U-Net.")
    parser.add_argument("--clean-dir", type=str, required=True, help="folder of clean photos (e.g. VOC2012 subset)")
    parser.add_argument("--masks-dir", type=str, required=True,
                         help="folder of masks from generate_synthetic_only.py (the generated/ output folder)")
    parser.add_argument("--blend-mode", type=str, choices=["screen", "multiply"], default="screen",
                         help="'screen' (default) = light/white damage marks (realistic for scratches/abrasion); "
                              "'multiply' = dark damage marks (realistic for soot/smut/heavy dirt)")
    parser.add_argument("--out-dir", type=str, default="./runs/stage1_restoration")
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--batch-size", type=int, default=8)
    parser.add_argument("--epochs", type=int, default=50)
    parser.add_argument("--lr", type=float, default=2e-4)
    parser.add_argument("--base-channels", type=int, default=64)
    parser.add_argument("--n-downsample", type=int, default=4)
    parser.add_argument("--num-workers", type=int, default=4)
    parser.add_argument("--save-every", type=int, default=5)
    parser.add_argument("--sample-every", type=int, default=200)
    parser.add_argument("--resume", type=str, default=None)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    parser.add_argument("--amp", action="store_true",
                         help="use automatic mixed precision (fp16) training -- meaningful speedup on modern "
                              "GPUs (T4 and newer) with minimal code cost; no effect on CPU")
    parser.add_argument("--steps-per-epoch", type=int, default=None,
                         help="if set, each 'epoch' samples this many random batches (with replacement) instead "
                              "of iterating the full dataset once. Use this to shorten epoch wall-clock time "
                              "without changing the model or data -- e.g. --steps-per-epoch 200 with batch-size "
                              "16 processes 3200 images/epoch instead of the full ~17000 in VOC2012, roughly "
                              "proportionally cutting epoch time.")
    parser.add_argument("--log-every", type=int, default=20,
                         help="print a data-loading-vs-compute timing breakdown every N steps, so you can see "
                              "whether slowness is coming from data loading (CPU/disk-bound) or the actual "
                              "model forward/backward pass (GPU-bound). Set to 0 to disable.")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    checkpoints_dir = os.path.join(args.out_dir, "checkpoints")
    samples_dir = os.path.join(args.out_dir, "samples")
    os.makedirs(checkpoints_dir, exist_ok=True)
    os.makedirs(samples_dir, exist_ok=True)

    device = torch.device(args.device)
    print(f"Using device: {device}")
    if device.type == "cuda":
        print(f"  GPU: {torch.cuda.get_device_name(device)}")
    cpu_count = os.cpu_count()
    print(f"  CPUs available: {cpu_count}, --num-workers set to {args.num_workers}")
    if args.num_workers > cpu_count:
        print(f"  Warning: --num-workers ({args.num_workers}) exceeds available CPUs ({cpu_count}); "
              f"this can hurt rather than help. Consider lowering it.")

    dataset = DegradedPairDataset(args.clean_dir, args.masks_dir, image_size=args.image_size, augment=True,
                                   blend_mode=args.blend_mode)
    print(f"Loaded {len(dataset)} clean images, {len(dataset.mask_files)} damage masks "
          f"from {args.clean_dir} / {args.masks_dir} (blend_mode={args.blend_mode})")
    if args.steps_per_epoch:
        num_samples = args.steps_per_epoch * args.batch_size
        sampler = RandomSampler(dataset, replacement=True, num_samples=num_samples)
        dataloader = DataLoader(dataset, batch_size=args.batch_size, sampler=sampler,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))
        print(f"Using --steps-per-epoch {args.steps_per_epoch}: each epoch samples "
              f"{num_samples} images (with replacement) instead of the full {len(dataset)}-image dataset")
    else:
        dataloader = DataLoader(dataset, batch_size=args.batch_size, shuffle=True,
                                 num_workers=args.num_workers, drop_last=True, pin_memory=(device.type == "cuda"),
                                 persistent_workers=(args.num_workers > 0))

    fixed_batch = next(iter(dataloader))

    model = RestorationUNet(
        in_channels=3, out_channels=3,
        base_channels=args.base_channels, n_downsample=args.n_downsample,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, betas=(0.9, 0.999))

    use_amp = args.amp and device.type == "cuda"
    if args.amp and device.type != "cuda":
        print("Note: --amp has no effect on CPU, ignoring.")
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    start_epoch = 1
    global_step = 0
    if args.resume:
        print(f"Resuming from {args.resume}")
        ckpt = torch.load(args.resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if "scaler_state_dict" in ckpt:
            scaler.load_state_dict(ckpt["scaler_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt.get("global_step", 0)

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model has {num_params:,} parameters")

    start_time = time.time()

    for epoch in range(start_epoch, args.epochs + 1):
        epoch_start = time.time()
        running_loss = 0.0
        running_data_time = 0.0
        running_compute_time = 0.0

        batch_end_time = time.time()  # marks the end of the previous iteration
        for batch in dataloader:
            data_time = time.time() - batch_end_time  # time spent waiting for this batch

            compute_start = time.time()
            damaged = batch["damaged"].to(device, non_blocking=True)
            clean = batch["clean"].to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.amp.autocast(device.type, enabled=use_amp):
                restored = model(damaged)
                loss, l1 = restoration_loss(restored, clean)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            if device.type == "cuda":
                torch.cuda.synchronize()  # so compute_time reflects actual GPU work, not just kernel launch
            compute_time = time.time() - compute_start

            running_loss += loss.item()
            running_data_time += data_time
            running_compute_time += compute_time
            global_step += 1

            if args.log_every and global_step % args.log_every == 0:
                msg = (f"  step {global_step}: data_time={data_time:.3f}s compute_time={compute_time:.3f}s "
                       f"({'data-loading-bound' if data_time > compute_time else 'compute-bound'})")
                if device.type == "cuda":
                    mem_alloc = torch.cuda.memory_allocated(device) / 1e9
                    mem_reserved = torch.cuda.memory_reserved(device) / 1e9
                    msg += f" | GPU mem: {mem_alloc:.2f}GB alloc / {mem_reserved:.2f}GB reserved"
                print(msg)

            if global_step % args.sample_every == 0:
                sample_path = os.path.join(samples_dir, f"step_{global_step:07d}.png")
                save_comparison_grid(model, fixed_batch, sample_path, device)

            batch_end_time = time.time()

        n_batches = len(dataloader)
        elapsed = time.time() - start_time
        print(f"[Epoch {epoch}/{args.epochs}] "
              f"loss={running_loss / n_batches:.4f} "
              f"avg_data_time={running_data_time / n_batches:.3f}s "
              f"avg_compute_time={running_compute_time / n_batches:.3f}s "
              f"epoch_time={format_duration(time.time() - epoch_start)} "
              f"total_elapsed={format_duration(elapsed)}")

        if epoch % args.save_every == 0 or epoch == args.epochs:
            ckpt_path = os.path.join(checkpoints_dir, f"stage1_epoch{epoch:04d}.pt")
            torch.save({
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict(),
                "args": vars(args),
            }, ckpt_path)
            print(f"  Saved checkpoint: {ckpt_path}")

    total_elapsed = time.time() - start_time
    print(f"\nTraining complete. Total time: {format_duration(total_elapsed)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile composite_damage.py
"""
Composite a generated damage mask (from generate_synthetic_only.py or
damage_generator.py) onto a clean target image, producing a damaged/clean
training pair for restoration model training.

The mask convention from this codebase: 255 = clean/undamaged, values toward
0 = damaged (dust, dirt, scratches etc).

Two blend modes are supported:
  - "screen" (default): LIGHTENS toward white at damaged pixels. This is the
    physically realistic choice for most scratch/abrasion damage, where the
    print's emulsion is scraped away and the lighter paper base shows
    through -- old photo scratches are usually bright/white marks, not dark
    ones.
  - "multiply": DARKENS toward black at damaged pixels. More appropriate for
    damage types that genuinely deposit dark material (soot/smut, heavy
    dirt, mold staining) rather than abrading the surface.

Since a single generated mask can currently mix multiple damage types
(e.g. scratches + smut) without tracking which pixel came from which type,
this is a per-composite choice rather than automatic per-pixel selection.
If your mask pool separates damage types into different files (e.g. by
generating scratches and smut as separate mask batches), you can composite
each with the blend mode that suits it and merge afterward, rather than
using one blend mode for a mixed mask.

Usage:
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png
    python composite_damage.py --clean path/to/clean.png --mask path/to/mask.png --out path/to/damaged.png --blend multiply
"""

import argparse
import cv2 as cv
import numpy as np


def composite(clean_img, mask_img, blend="screen"):
    if clean_img.shape[:2] != mask_img.shape[:2]:
        mask_img = cv.resize(mask_img, (clean_img.shape[1], clean_img.shape[0]), interpolation=cv.INTER_LINEAR)

    mask_norm = mask_img.astype(np.float32) / 255.0
    if clean_img.ndim == 3 and mask_norm.ndim == 2:
        mask_norm = mask_norm[:, :, None]

    clean_f = clean_img.astype(np.float32)

    if blend == "screen":
        # Lightens toward white at damaged (low-mask) pixels.
        damaged = 255.0 - (255.0 - clean_f) * mask_norm
    elif blend == "multiply":
        # Darkens toward black at damaged (low-mask) pixels.
        damaged = clean_f * mask_norm
    else:
        raise ValueError(f"Unknown blend mode '{blend}', expected 'screen' or 'multiply'")

    return np.clip(damaged, 0, 255).astype(np.uint8)


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Composite a damage mask onto a clean image.')
    parser.add_argument('--clean', required=True, help='path to the clean input image')
    parser.add_argument('--mask', required=True, help='path to the generated grayscale damage mask')
    parser.add_argument('--out', required=True, help='path to write the damaged output image')
    parser.add_argument('--blend', choices=['screen', 'multiply'], default='screen',
                         help="'screen' (default) produces light/white damage marks; "
                              "'multiply' produces dark damage marks")
    args = parser.parse_args()

    clean_img = cv.imread(args.clean, cv.IMREAD_UNCHANGED)
    mask_img = cv.imread(args.mask, cv.IMREAD_GRAYSCALE)

    damaged = composite(clean_img, mask_img, blend=args.blend)
    cv.imwrite(args.out, damaged)
    print(f"Wrote damaged image to {args.out} (blend={args.blend})")


## 4. Get VOC2012 clean images — automatic, no manual download

Same as before: `torchvision.datasets.VOCDetection(..., download=True)` fetches and extracts it. Requires **Internet: On** in Settings (step 0 above) or this cell will fail with a network error.

Downloads to `/kaggle/working/voc_data` -- if you want this to persist without re-downloading every session, consider uploading it once as a Kaggle Dataset and attaching it instead (Add Data), then skip this cell in future sessions.

In [ ]:
import torchvision.datasets as tvds

VOC_ROOT = '/kaggle/working/voc_data'
VOC_JPEG_DIR = os.path.join(VOC_ROOT, 'VOCdevkit', 'VOC2012', 'JPEGImages')

if os.path.isdir(VOC_JPEG_DIR) and len(os.listdir(VOC_JPEG_DIR)) > 0:
    print(f'VOC2012 already present at {VOC_JPEG_DIR}, skipping download.')
else:
    os.makedirs(VOC_ROOT, exist_ok=True)
    _ = tvds.VOCDetection(root=VOC_ROOT, year='2012', image_set='train', download=True)

num_images = len([f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')])
print(f'VOC2012 ready: {num_images} images at {VOC_JPEG_DIR}')


## 5. Build a 100-image subset (for the quick test in section 7)


In [ ]:
import random, shutil

SUBSET_DIR = '/kaggle/working/voc_subset_100/images'
os.makedirs(SUBSET_DIR, exist_ok=True)

existing = [f for f in os.listdir(SUBSET_DIR) if f.lower().endswith('.jpg')]
if len(existing) >= 100:
    subset = existing[:100]
    print(f'Subset already present ({len(existing)} images) at {SUBSET_DIR}, skipping copy.')
else:
    all_voc_images = [f for f in os.listdir(VOC_JPEG_DIR) if f.lower().endswith('.jpg')]
    random.seed(42)
    subset = random.sample(all_voc_images, min(100, len(all_voc_images)))
    for fname in subset:
        shutil.copy(os.path.join(VOC_JPEG_DIR, fname), os.path.join(SUBSET_DIR, fname))
    print(f'Copied {len(subset)} images to {SUBSET_DIR}')


## 6. Generate damage masks with FilmDamageSimulator

Clones the simulator (for its `synthetic/` damage-patch assets) and recreates `generate_synthetic_only.py` — builds masks from ONLY the classified synthetic patches, never touching the real scanned frames.

In [ ]:
import os
if os.path.isdir('FilmDamageSimulator'):
    print('FilmDamageSimulator already cloned, skipping.')
else:
    !git clone --depth 1 https://github.com/daniela997/FilmDamageSimulator.git


In [ ]:
%%writefile FilmDamageSimulator/damage_generator/generate_synthetic_only.py
"""
Generate damage overlay masks using ONLY the pre-classified synthetic damage
patches in /synthetic/<type>/ (e.g. scratches, smut), without ever touching
the real scanned film frames in /scans/.

This bypasses damage_generator.py's default behaviour, which always loads
/scans/ and mixes real scanned artifact crops into the sampling pool even
when --synthetic is passed. Here, only the folder(s) you name are loaded,
and artifact count/size statistics are fit on those patches' own area
distribution instead of the real-scan-derived Gamma distributions.

Usage:
    python generate_synthetic_only.py --types scratches,smut --height 1024 --width 1024
    python generate_synthetic_only.py --types scratches --procedural-scratches
"""

import os
import argparse
import uuid
import random
import numpy as np
import pandas as pd
import cv2 as cv
import scipy.stats as stats
import skimage.transform as skimage_tf

from scans import load_images
from generate_masks import generate_perlin_noise_2d, increase_contrast, random_perlin_with_numpy, line_scratch


def sample_size_from_own_distribution(df, num_artifact):
    """Fit a Gamma distribution to this dataframe's OWN artifact areas
    (instead of a real-scan-derived one) and sample target sizes from it."""
    areas = df['Contour Area']
    gamma_param = stats.gamma.fit(areas, floc=0)
    shape, _, scale = gamma_param
    return np.random.gamma(shape, scale, num_artifact)


def sample_closest_in_area(df, target_areas):
    df = df.sample(frac=1).reset_index(drop=True)
    areas = df['Contour Area']
    indexes = []
    for target in target_areas:
        candidates = df.iloc[(areas - target).abs().argsort()[:15]].index.tolist()
        index = random.choice(candidates)
        indexes.append(index)
        areas = areas.drop(areas.index[[index]])
    picked = df.iloc[indexes].copy()
    picked['Target size'] = target_areas
    return picked


def build_mask(target_size, per_type_dfs, per_type_counts, rescale=True, verbose=False):
    rescale_factor = (target_size[0] / 2560 if target_size[0] <= target_size[1]
                       else target_size[1] / 2560) if rescale else 1.

    selected_frames = []
    for artifact_type, df in per_type_dfs.items():
        lo, hi = per_type_counts[artifact_type]
        num = int(np.random.randint(lo, hi + 1))
        if num == 0 or len(df) == 0:
            continue
        target_areas = sample_size_from_own_distribution(df, num)
        picked = sample_closest_in_area(df, target_areas)
        selected_frames.append(picked)
        if verbose:
            print(f"Selected {num} '{artifact_type}' artifacts")

    if not selected_frames:
        raise ValueError("No artifacts selected - check your --types and --min-count/--max-count")

    selected_artifacts_df = pd.concat(selected_frames, ignore_index=True)
    artifacts_num = len(selected_artifacts_df)

    mask_final = np.zeros(target_size).astype(np.uint8)
    perlin_noise = generate_perlin_noise_2d(target_size, (2, 2))
    normalised_noise = (perlin_noise - np.min(perlin_noise)) / np.ptp(perlin_noise)
    xs, ys = random_perlin_with_numpy(artifacts_num, normalised_noise)
    random_angles = np.random.randint(0, 360, size=artifacts_num)

    i = 0
    for _, artifact_row in selected_artifacts_df.iterrows():
        try:
            artifact = artifact_row['Artifact'].astype(np.uint8)
            random_scale = artifact_row['Target size'] / artifact_row['Contour Area']
            random_angle = random_angles[i]
            new_rescale_factor = rescale_factor * np.sqrt(random_scale)
            artifact = skimage_tf.rescale(artifact, round(new_rescale_factor, 2), anti_aliasing=True, preserve_range=True)
            artifact = skimage_tf.rotate(artifact, angle=random_angle, resize=True, preserve_range=True)
            artifact_w, artifact_h = artifact.shape[:2]

            x1 = xs[i] - artifact_w // 2
            x2 = x1 + artifact_w
            if x1 < 0:
                artifact = artifact[-x1:, :]; x1 = 0
            if x2 > target_size[0]:
                artifact = artifact[:-(x2 - target_size[0]), :]; x2 = target_size[0]

            y1 = ys[i] - artifact_h // 2
            y2 = y1 + artifact_h
            if y1 < 0:
                artifact = artifact[:, -y1:]; y1 = 0
            if y2 > target_size[1]:
                artifact = artifact[:, :-(y2 - target_size[1])]; y2 = target_size[1]

            mask_final[x1:x2, y1:y2] = np.where(
                artifact > mask_final[x1:x2, y1:y2], artifact, mask_final[x1:x2, y1:y2]
            )
            i += 1
        except Exception:
            i += 1
            continue

    mask_final = np.invert(mask_final.astype(np.uint8))
    binarised = ((mask_final > 240) * 255).astype(np.uint8)
    return mask_final.astype(np.uint8), binarised


def add_procedural_scratches(mask, height, width, verbose=False):
    """Blend in fully procedural (Perlin-noise-based) scratch lines.
    These require NO source images at all -- real or synthetic -- so they
    are always 'safe' to include without pulling in any scan data."""
    num_extra_scratch = int(np.random.gamma(6, 2, 1)[0])
    for _ in range(num_extra_scratch):
        length = np.random.randint(10, high=max(height, width), dtype=int)
        try:
            scratch = line_scratch(np.array(length))
            sw, sh = scratch.shape[:2]
            if sw >= width or sh >= height:
                continue
            x1 = np.random.randint(0, width - sw)
            y1 = np.random.randint(0, height - sh)
            region = mask[x1:x1 + sw, y1:y1 + sh]
            mask[x1:x1 + sw, y1:y1 + sh] = np.minimum(region, np.invert(scratch.astype(np.uint8)))
        except Exception:
            continue
    if verbose:
        print(f"Added {num_extra_scratch} procedural scratch lines")
    return mask


if __name__ == '__main__':
    parser = argparse.ArgumentParser(
        description='Generate damage masks from ONLY classified synthetic patches (no scanned frames).'
    )
    parser.add_argument('--types', type=str, default='scratches,smut',
                         help='comma-separated subfolder names under /synthetic/, '
                              'e.g. scratches,smut,dirt,dots,hair,hair-short,lint,sprinkles,spots,stain')
    parser.add_argument('--height', type=int, default=1024)
    parser.add_argument('--width', type=int, default=1024)
    parser.add_argument('--min-count', type=int, default=3, help='min number of artifacts per type')
    parser.add_argument('--max-count', type=int, default=15, help='max number of artifacts per type')
    parser.add_argument('--procedural-scratches', action='store_true',
                         help='also blend in fully procedural line scratches (no source image needed)')
    parser.add_argument('--n', type=int, default=1, help='how many masks to generate')
    parser.add_argument('--verbose', action='store_true')
    args = parser.parse_args()

    abs_path = os.path.abspath(os.path.dirname(__file__))
    synthetic_path = os.path.dirname(os.path.normpath(abs_path)) + '/synthetic/'
    out_dir = os.path.dirname(os.path.normpath(abs_path)) + '/generated/'
    os.makedirs(out_dir, exist_ok=True)

    types = [t.strip() for t in args.types.split(',') if t.strip()]

    per_type_dfs = {}
    for t in types:
        df = load_images(synthetic_path, t, verbose=args.verbose)
        df['Contour Area'] = df['Non-zero pixel area']
        per_type_dfs[t] = df
        print(f"Loaded {len(df)} '{t}' artifact patches from /synthetic/{t}/")

    per_type_counts = {t: (args.min_count, args.max_count) for t in types}

    for n in range(args.n):
        mask, binary_mask = build_mask(
            (args.height, args.width), per_type_dfs, per_type_counts, verbose=args.verbose
        )

        if args.procedural_scratches:
            mask = add_procedural_scratches(mask, args.height, args.width, verbose=args.verbose)
            binary_mask = ((mask > 240) * 255).astype(np.uint8)

        uid = str(uuid.uuid4())[:8]
        tag = "_".join(types)
        cv.imwrite(out_dir + f'mask_{tag}_{uid}.png', mask)
        cv.imwrite(out_dir + f'binarised_mask_{tag}_{uid}.png', binary_mask)
        print(f"[{n+1}/{args.n}] Saved mask_{tag}_{uid}.png")

    print(f"Done. Masks written to {out_dir}")


In [ ]:
MASKS_DIR = '/kaggle/working/generated_masks'
os.makedirs(MASKS_DIR, exist_ok=True)

TARGET_N_MASKS = 60
existing_masks = [f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')]

if len(existing_masks) >= TARGET_N_MASKS:
    print(f'{len(existing_masks)} masks already present at {MASKS_DIR}, skipping generation.')
else:
    os.chdir('/kaggle/working/FilmDamageSimulator/damage_generator')
    !python generate_synthetic_only.py --types scratches,smut \
        --height 256 --width 256 --min-count 3 --max-count 15 --n {TARGET_N_MASKS} --verbose
    os.chdir('/kaggle/working')

    src_dir = 'FilmDamageSimulator/generated'
    for fname in os.listdir(src_dir):
        shutil.copy(os.path.join(src_dir, fname), os.path.join(MASKS_DIR, fname))

n_masks = len([f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')])
print(f'{n_masks} usable masks ready at {MASKS_DIR}')


## 7. Test: apply damage masks to a few photos

Visual sanity check — composites real VOC photos with real generated masks and displays [clean | damaged] side by side. Damage should render light/white (screen blend, the default) not dark.

In [ ]:
import matplotlib.pyplot as plt
import cv2 as cv
from composite_damage import composite

sample_clean_files = subset[:4]
mask_files = [f for f in os.listdir(MASKS_DIR) if f.startswith('mask_')]
sample_mask_files = random.sample(mask_files, 4)

fig, axes = plt.subplots(4, 2, figsize=(6, 12))
for i, (clean_fname, mask_fname) in enumerate(zip(sample_clean_files, sample_mask_files)):
    clean_img = cv.imread(os.path.join(SUBSET_DIR, clean_fname))
    mask_img = cv.imread(os.path.join(MASKS_DIR, mask_fname), cv.IMREAD_GRAYSCALE)
    damaged_img = composite(clean_img, mask_img)  # default blend='screen'

    axes[i, 0].imshow(cv.cvtColor(clean_img, cv.COLOR_BGR2RGB))
    axes[i, 0].set_title('Clean')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(cv.cvtColor(damaged_img, cv.COLOR_BGR2RGB))
    axes[i, 1].set_title('Damaged (mask applied)')
    axes[i, 1].axis('off')

plt.tight_layout()
plt.show()


## 8. Test: train Stage 1 on the 100-image subset

Short run purely to confirm the full pipeline works end to end on this runtime before committing to a long unattended run.

In [ ]:
!python train_stage1_restoration.py \
    --clean-dir "$SUBSET_DIR" \
    --masks-dir "$MASKS_DIR" \
    --epochs 10 \
    --batch-size 8 \
    --image-size 128 \
    --num-workers 2 \
    --sample-every 10 \
    --save-every 5 \
    --out-dir ./runs/stage1_smoke_test \
    --device cuda


## 9. View a reconstruction sample

Top row = damaged input, middle row = model output, bottom row = clean target.

In [ ]:
import glob
from PIL import Image

sample_files = sorted(glob.glob('runs/stage1_smoke_test/samples/*.png'))
if sample_files:
    img = Image.open(sample_files[-1])
    plt.figure(figsize=(14, 7))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Latest sample: {sample_files[-1]}')
    plt.show()
else:
    print('No samples found yet — check the training cell above ran successfully.')


## 10. Baseline sanity check: short run at full resolution

Before committing to a long full training run: 8 epochs at your REAL target resolution/batch size, on the full VOC dataset (not the 100-image subset). Confirms loss decreases under real conditions and gives you a wall-clock time-per-epoch estimate.


In [ ]:
!python train_stage1_restoration.py \
    --clean-dir "$VOC_JPEG_DIR" \
    --masks-dir "$MASKS_DIR" \
    --epochs 8 \
    --batch-size 16 \
    --image-size 256 \
    --num-workers 2 \
    --sample-every 50 \
    --save-every 4 \
    --amp \
    --out-dir ./runs/baseline_check \
    --device cuda

# Note the printed epoch_time to estimate full-run duration, e.g.
# if epoch_time ~= 40s and you plan --epochs 100, that's roughly 1.1 hours total.


## 11. Full training run

Once the baseline check above looks right, scale up. Kaggle GPU sessions cap at 12 hours and you get 30 GPU-hours/week -- use `--steps-per-epoch` to control how much fits in one sitting, and remember to **Save Version** before the session ends so `/kaggle/working/runs/...` is preserved as this version's Output.

To resume in a later session: attach this notebook's previous Output as an input dataset, copy the latest checkpoint from `/kaggle/input/.../checkpoints/` into `/kaggle/working/runs/stage1_restoration/checkpoints/`, then pass it to `--resume` below.


In [ ]:
# Example full run:
# !python train_stage1_restoration.py \
#     --clean-dir "$VOC_JPEG_DIR" \
#     --masks-dir "$MASKS_DIR" \
#     --epochs 100 \
#     --batch-size 16 \
#     --image-size 256 \
#     --num-workers 2 \
#     --amp \
#     --out-dir ./runs/stage1_restoration \
#     --device cuda

# To resume from a previous session's checkpoint:
# !python train_stage1_restoration.py \
#     --clean-dir "$VOC_JPEG_DIR" \
#     --masks-dir "$MASKS_DIR" \
#     --epochs 100 \
#     --batch-size 16 \
#     --image-size 256 \
#     --amp \
#     --out-dir ./runs/stage1_restoration \
#     --device cuda \
#     --resume /kaggle/working/runs/stage1_restoration/checkpoints/<latest>.pt


## 12. Evaluate a checkpoint: PSNR/SSIM + visual comparison

Quantitative check of how a checkpoint is actually doing, not just eyeballing sample grids. Computes PSNR/SSIM for damaged-vs-clean (the do-nothing baseline) and restored-vs-clean (the model), so you can see the actual gain (or lack of one, if the checkpoint is still early).

Reconstructs the model architecture automatically from the checkpoint itself -- no need to match --base-channels/--n-downsample by hand.

Note: for a real baseline sanity check, evaluating on training images is fine. For your FINAL cross-method thesis comparison, use a held-out clean-image set that was never used in any training stage.


In [ ]:
%%writefile evaluate.py
"""
Evaluate a trained Stage 1 restoration checkpoint: computes PSNR and SSIM
between (a) damaged vs. clean and (b) restored vs. clean, so you can see
the actual improvement the model provides, not just eyeball sample grids.

Reconstructs the model architecture from the checkpoint's saved args
(base_channels, n_downsample) automatically -- no need to re-specify them
and risk a mismatch.

Note: for a real baseline sanity check, evaluating on images the model saw
during training is fine (you're just confirming the pipeline/model works).
For your FINAL thesis comparison across all three restoration methods, use
a held-out clean-image set that was never used in ANY training stage --
see the pre-training checklist from earlier in this project.

Usage:
    python evaluate.py --checkpoint ./runs/baseline_check/checkpoints/stage1_epoch0008.pt \
        --clean-dir ./voc_subset_100/images --masks-dir ./generated_masks \
        --num-samples 20 --out-dir ./eval_results
"""

import argparse
import os

import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
import torchvision.utils as vutils

from models.restoration_net import RestorationUNet
from data.degraded_pair_dataset import DegradedPairDataset, denormalize


def tensor_to_numpy_image(tensor: torch.Tensor) -> np.ndarray:
    """(C, H, W) tensor in [0, 1] -> (H, W, C) numpy array in [0, 255] uint8."""
    arr = tensor.clamp(0, 1).mul(255).byte().permute(1, 2, 0).cpu().numpy()
    return arr


def main():
    parser = argparse.ArgumentParser(description="Evaluate a Stage 1 checkpoint with PSNR/SSIM.")
    parser.add_argument("--checkpoint", type=str, required=True)
    parser.add_argument("--clean-dir", type=str, required=True)
    parser.add_argument("--masks-dir", type=str, required=True)
    parser.add_argument("--num-samples", type=int, default=20,
                         help="how many random (clean, mask) pairs to evaluate on")
    parser.add_argument("--image-size", type=int, default=None,
                         help="defaults to the image size the checkpoint was trained with")
    parser.add_argument("--out-dir", type=str, default="./eval_results")
    parser.add_argument("--seed", type=int, default=123)
    parser.add_argument("--device", type=str, default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()

    os.makedirs(args.out_dir, exist_ok=True)
    device = torch.device(args.device)

    print(f"Loading checkpoint: {args.checkpoint}")
    ckpt = torch.load(args.checkpoint, map_location=device)
    train_args = ckpt.get("args", {})

    image_size = args.image_size or train_args.get("image_size", 256)
    base_channels = train_args.get("base_channels", 64)
    n_downsample = train_args.get("n_downsample", 4)
    blend_mode = train_args.get("blend_mode", "screen")

    print(f"  Reconstructed from checkpoint: image_size={image_size}, base_channels={base_channels}, "
          f"n_downsample={n_downsample}, blend_mode={blend_mode}")
    print(f"  Checkpoint was saved at epoch {ckpt.get('epoch', '?')}, step {ckpt.get('global_step', '?')}")

    model = RestorationUNet(in_channels=3, out_channels=3, base_channels=base_channels,
                             n_downsample=n_downsample).to(device)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    torch.manual_seed(args.seed)
    dataset = DegradedPairDataset(args.clean_dir, args.masks_dir, image_size=image_size,
                                   augment=False, blend_mode=blend_mode)
    num_samples = min(args.num_samples, len(dataset))
    indices = torch.randperm(len(dataset))[:num_samples].tolist()
    print(f"Evaluating on {num_samples} samples (seed={args.seed})")

    damaged_psnrs, damaged_ssims = [], []
    restored_psnrs, restored_ssims = [], []

    comparison_rows = []

    with torch.no_grad():
        for i, idx in enumerate(indices):
            sample = dataset[idx]
            damaged = sample["damaged"].unsqueeze(0).to(device)
            clean = sample["clean"].unsqueeze(0).to(device)

            restored = model(damaged)

            damaged_np = tensor_to_numpy_image(denormalize(damaged[0]))
            clean_np = tensor_to_numpy_image(denormalize(clean[0]))
            restored_np = tensor_to_numpy_image(denormalize(restored[0]))

            d_psnr = peak_signal_noise_ratio(clean_np, damaged_np, data_range=255)
            d_ssim = structural_similarity(clean_np, damaged_np, channel_axis=2, data_range=255)
            r_psnr = peak_signal_noise_ratio(clean_np, restored_np, data_range=255)
            r_ssim = structural_similarity(clean_np, restored_np, channel_axis=2, data_range=255)

            damaged_psnrs.append(d_psnr)
            damaged_ssims.append(d_ssim)
            restored_psnrs.append(r_psnr)
            restored_ssims.append(r_ssim)

            if i < 6:  # keep a handful for a visual comparison grid
                comparison_rows.append((denormalize(damaged[0]).cpu(),
                                         denormalize(restored[0]).cpu(),
                                         denormalize(clean[0]).cpu()))

    def summarize(name, values):
        arr = np.array(values)
        print(f"  {name}: mean={arr.mean():.3f}  std={arr.std():.3f}  min={arr.min():.3f}  max={arr.max():.3f}")

    print("\n=== Damaged vs. Clean (baseline, no restoration) ===")
    summarize("PSNR (dB, higher=better)", damaged_psnrs)
    summarize("SSIM (0-1, higher=better)", damaged_ssims)

    print("\n=== Restored vs. Clean (model output) ===")
    summarize("PSNR (dB, higher=better)", restored_psnrs)
    summarize("SSIM (0-1, higher=better)", restored_ssims)

    psnr_gain = np.mean(restored_psnrs) - np.mean(damaged_psnrs)
    ssim_gain = np.mean(restored_ssims) - np.mean(damaged_ssims)
    print(f"\n=== Improvement from restoration ===")
    print(f"  PSNR gain: {psnr_gain:+.3f} dB")
    print(f"  SSIM gain: {ssim_gain:+.3f}")
    if psnr_gain <= 0:
        print("  Note: non-positive PSNR gain means the model isn't yet improving over doing nothing -- "
              "expected for a short baseline check, but worth watching on longer runs.")

    # Save per-sample results to CSV
    csv_path = os.path.join(args.out_dir, "metrics.csv")
    with open(csv_path, "w") as f:
        f.write("sample_index,damaged_psnr,damaged_ssim,restored_psnr,restored_ssim\n")
        for idx, dp, ds, rp, rs in zip(indices, damaged_psnrs, damaged_ssims, restored_psnrs, restored_ssims):
            f.write(f"{idx},{dp:.4f},{ds:.4f},{rp:.4f},{rs:.4f}\n")
    print(f"\nPer-sample metrics saved to {csv_path}")

    # Save a visual comparison grid
    if comparison_rows:
        damaged_imgs = torch.stack([r[0] for r in comparison_rows])
        restored_imgs = torch.stack([r[1] for r in comparison_rows])
        clean_imgs = torch.stack([r[2] for r in comparison_rows])
        grid = torch.cat([damaged_imgs, restored_imgs, clean_imgs], dim=0)
        grid_path = os.path.join(args.out_dir, "comparison_grid.png")
        vutils.save_image(grid, grid_path, nrow=len(comparison_rows))
        print(f"Visual comparison grid saved to {grid_path}")


if __name__ == "__main__":
    main()


In [ ]:
!python evaluate.py \
    --checkpoint ./runs/baseline_check/checkpoints/stage1_epoch0008.pt \
    --clean-dir "$VOC_JPEG_DIR" \
    --masks-dir "$MASKS_DIR" \
    --num-samples 30 \
    --out-dir ./eval_results \
    --device cuda


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

img = Image.open('eval_results/comparison_grid.png')
plt.figure(figsize=(16, 8))
plt.imshow(img)
plt.axis('off')
plt.title('Top: damaged | Middle: restored | Bottom: clean')
plt.show()
